In [1]:
# 학번과 이름을 출력하세요
# 예시
# 학번 = '30261240'
# 이름 = '이현정'

학번 = '20231972'
이름 = '김기웅'

print(학번, 이름)

20231972 김기웅


In [2]:
class MyStack:
    def pop(self):
        return "Removed"

def remove_last(obj):
    return obj.pop()

print(remove_last([1, 2, 3]))
print(remove_last(MyStack()))

3
Removed


In [4]:
class MySequence:
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index]

seq = MySequence([10, 20, 30])

print(len(seq))
print(seq[1])

for item in seq:
    print(item)

3
20
10
20
30


In [6]:
class Foo:
    def __getitem__(self, pos):
        return range(0, 30, 10)[pos]

In [8]:
f = Foo()
f[1]

10

In [10]:
for i in f: print(i)

0
10
20


In [12]:
20 in f

True

In [14]:
15 in f

False

In [21]:
import collections

Card = collections.namedtuple('Card', ['rank', 'suit'])

class FrenchDeck:
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [Card(rank, suit) for suit in self.suits for rank in self.ranks]

    def __len__(self):
        return len(self._cards)

    def __getitem__(self, position):
        return self._cards[position]

In [23]:
from random import shuffle
deck = FrenchDeck()
shuffle(deck)

TypeError: 'FrenchDeck' object does not support item assignment

In [27]:
def set_card(deck, position, card):
    deck._cards[position] = card
FrenchDeck.__setitem__ = set_card

In [29]:
shuffle(deck)
deck[:5]

[Card(rank='K', suit='hearts'),
 Card(rank='Q', suit='hearts'),
 Card(rank='10', suit='spades'),
 Card(rank='3', suit='hearts'),
 Card(rank='7', suit='diamonds')]

In [32]:
import collections
import collections.abc

Card = collections.namedtuple('Card', ['rank', 'suit'])

class FrenchDeck(collections.abc.MutableSequence):
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [Card(rank, suit) for suit in self.suits for rank in self.ranks]

    def __len__(self):
        return len(self._cards)

    def __getitem__(self, position):
        return self._cards[position]

    def __setitem__(deck, position, card):
        deck._cards[position] = card

    def __delitem__(self, position):
        del self._cards[position]

    def insert(self, position, value):
        self._cards.insert(position, value)

In [34]:
from abc import ABC, abstractmethod

class Drawable(ABC):
    @abstractmethod
    def draw(self): pass

class ExternalPainter:
    def draw(self):
        print("Drawing")

Drawable.register(ExternalPainter)

obj = ExternalPainter()

print(isinstance(obj, Drawable))

True


In [36]:
from collections.abc import Sized

class MyCollection:
    def __len__(self):
        return 10

Sized.register(MyCollection)

__main__.MyCollection

In [38]:
import abc

class Tombola(abc.ABC):

    @abc.abstractmethod
    def load(self, iterable):
        """Add items from an iterable."""

    @abc.abstractmethod
    def pick(self):
        """Remove item at random, returning it.
        
        This method should raise `LookupError` when the instance is Empty.
        """


    @abc.abstractmethod
    def load(self):
        """Return `True` if there's at least 1 item, `False` otherwise."""
        return bool(self.inspect())

    @abc.abstractmethod
    def inspect(self):
        """Return a sorted tuple woth the items currently inside."""
        items = []
        while True:
            try:
                item.append(self.pick())
            except LookupError:
                break
        self.load(items)
        return tuple(sorted(items))
    

In [40]:
import random

class BingoCage(Tombola):
    def __init__(self, items):
        self._randomizer = random.SystemRandom()
        self._items = []
        self.load(items)

    def load(self, items):
        self._items.extend(items)
        self._randomizer.shuffle(self._items)

    def pick(self):
        try:
            return self._items.pop()
        except IndexError:
            raise LookupError('pick from empty BingoCage')

    def __call__(self):
        self.pick()

In [42]:
import random

class LotteryBlower(Tombola):

    def __init__(self, iterable):
        self._balls = list(iterable)

    def load(self, iterable):
        self._balls.extend(iterable)

    def pick(self):
        try:
            position = random.randrange(len(self._balls))
        except ValueError:
            raise LookupError('pick from empty BingoCage')
        return self._balls.pop(position)

    def loaded(self):
        return bool(self._balls)

    def inspect(self):
        return tuple(sorted(self._balls))

In [44]:
from random import randrange

@Tombola.register
class TomboList(list):

    def pick(self):
        if self:
            position = randrange(len(self))
            return self.pop(position)

        else:
            raise LookupError('pop from empty TomboList')

    load = list.extend

    def loaded(self):
        return bool(self)

    def inspect(self):
        return tuple(sorted(self))

In [ ]:
import doctest

TEST_FILE = 'tombola_tests.rst'
TEST_MSG = '{0:16} {1.attempted:2} tests, {1.failed:2} failed - {2}'

def main(argv):
    verbose = '-v' in argv
    real_subclasses = Tombola.__subclasses__()
    virtual_subclasses = list(Tombola._abc_registry)

    for cls in real_subclasses + virtual_subclasses:
        test(cls, verbose)

def test(cls, verbose=False):
    res = doctest.testfile(
        TEST_FILE,
        globs={'ConcreteTombola': cls},
        verbose=verbose,
        optionflags=doctest.REPORT_ONLY_FIRST_FAILURE
    )
    tag = 'FAIL' if res.failed else 'OK'
    print(TEST_MSG.format(cls.__name__, res, tag))

if __name__ == '__main__':
    import sys
    main(sys.argv)


AttributeError: type object 'Tombola' has no attribute '_abc_registry'